In [1]:
import os
import json
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from torchvision import transforms
from torchvision.datasets import CIFAR10
import torch.utils.data as data
from torch.utils.data import DataLoader, Dataset
import matplotlib.pyplot
import matplotlib
matplotlib.rcParams["lines.linewidth"] = 2.0
!pip install pytorch_lightning




In [2]:
import pytorch_lightning as pl
from pytorch_lightning.callbacks import ModelCheckpoint, LearningRateMonitor
!pip install torch_geometric
import torch_geometric
import seaborn as sns
sns.set_style("whitegrid")


def set_seed(seed):
  np.random.seed(seed)
  torch.manual_seed(seed)
  torch.cuda.manual_seed_all(seed)
set_seed(42)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device("cuda:0")if torch.cuda.is_available()else torch.device

In [3]:
import torch_geometric.nn as geom_nn
import torch_geometric.data as geom_data

In [4]:
data_path = "data_path"
checkpoint_path = "checkpoint_path"

In [5]:
import urllib.request
from urllib.error import HTTPError

base_url = "https://raw.githubusercontent.com/phlippe/saved_models/main/tutorial7/"

pretrained_filename =  ["NodeLevelMLP.ckpt", "NodeLevelGNN.ckpt", "GraphLevelGraphConv.ckpt"]
os.makedirs(checkpoint_path, exist_ok = True)

def pretrained_file(base_url:str, pretrained_filename:str):
 for file_name in pretrained_filename:
  file_path = os.path.join(checkpoint_path, file_name)
  if "/" in file_name:
      os.makedirs(os.path.dirname(file_path), exist_ok = True)
  if  not os.path.isfile(file_path):
        file_url = base_url + file_name
        print(f"downloading this file_url{file_url}")
        try:
          urllib.request.urlretrieve(file_url, file_path)
        except HTTPError as e:
         print("downloading this filename from pretrained_file/n", e)


if __name__ == "__main__":
  pretrained_file(base_url, pretrained_filename)

In [6]:
class GCN(nn.Module):
  def __init__(self, c_in, c_out):
    super().__init__()
    self.projection = nn.Linear(c_in, c_out)



  def forward(self, node_feats, adj_matrix):
    num_neighbour = torch.sum(adj_matrix, dim = -1, keepdim = True)
    node_feats = self.projection(node_feats)
    node_feats = torch.bmm(adj_matrix, node_feats)
    node_feats = node_feats/num_neighbour
    return node_feats

In [7]:
node_feats = torch.arange(8 ,dtype = torch.float32).view(1, 4, 2)
adj_matrix = torch.Tensor([[[1, 0, 0, 1],
                            [1, 0, 1, 0],
                            [1, 0, 1, 0],
                            [1,0, 0, 1]]])

print(f"the node_feats/n", node_feats)
print(f"the adj_matrix/n", adj_matrix)

the node_feats/n tensor([[[0., 1.],
         [2., 3.],
         [4., 5.],
         [6., 7.]]])
the adj_matrix/n tensor([[[1., 0., 0., 1.],
         [1., 0., 1., 0.],
         [1., 0., 1., 0.],
         [1., 0., 0., 1.]]])


In [8]:
layer = GCN(c_in = 2, c_out = 4)
layer.projection.weight.data = torch.tensor([[1, 0, 0, 1], [0, 1, 0, 1]], dtype=torch.float32).T
layer.projection.bias.data = torch.tensor([0, 0, 1, 0], dtype=torch.float32)
with torch.no_grad():
  out_feats = layer(node_feats, adj_matrix)

  print("out_feat\n", out_feats)
  print("node_feats\n", node_feats)
  print("adj_matrix\n", adj_matrix)

out_feat
 tensor([[[3., 4., 1., 7.],
         [2., 3., 1., 5.],
         [2., 3., 1., 5.],
         [3., 4., 1., 7.]]])
node_feats
 tensor([[[0., 1.],
         [2., 3.],
         [4., 5.],
         [6., 7.]]])
adj_matrix
 tensor([[[1., 0., 0., 1.],
         [1., 0., 1., 0.],
         [1., 0., 1., 0.],
         [1., 0., 0., 1.]]])


# Graph - Attention - layer

In [9]:
class GAT(nn.Module):
  def __init__(self, c_in, c_out, num_heads = 4, concat_head = True, alpha = 0.5):
    super().__init__()

    self.num_heads = num_heads
    self.concat_head = concat_head
    if self.concat_head:
      assert c_out % num_heads == 0, "c_out should be divisible by num_heads"
      c_out = c_out // num_heads



    self.projection = nn.Linear(c_in, c_out * num_heads)
    self.a = nn.Parameter(torch.randn(num_heads, 2 * c_out))


    nn.init.xavier_uniform_(self.projection.weight.data, gain = 1.414)
    nn.init.xavier_uniform_(self.a.data, gain = 1.414)
    self.leaky_relu = nn.LeakyReLU(alpha)



  def forward(self, node_feats, adj_matrix, print_attr = False):
    batch_size, num_nodes = node_feats.size(0), node_feats.size(1)
    node_feats = self.projection(node_feats)
    node_feats = node_feats.view(batch_size, num_nodes, self.num_heads, -1)
    edges = adj_matrix.nonzero(as_tuple = False)
    node_feats_flat = node_feats.view(batch_size * num_nodes, self.num_heads, -1)
    # Corrected index calculation: batch_idx * num_nodes + node_idx
    edge_indices_rows = edges[:,0] * num_nodes + edges[:, 1]
    edge_indices_cols = edges[:, 0] * num_nodes + edges[:, 2]
    input_value = torch.cat([torch.index_select(input = node_feats_flat, index = edge_indices_rows, dim = 0),
                             torch.index_select(input = node_feats_flat, index = edge_indices_cols, dim = 0)], dim = -1)

    attn_logit = torch.einsum("bhc,hc->bh",input_value, self.a)
    attn_logit = self.leaky_relu(attn_logit)

    adjacent_matrixes = attn_logit.new_zeros(adj_matrix.shape+(self.num_heads,)).fill_(-9e-15)
    adjacent_matrixes[adj_matrix[...,None].repeat(1,1,1,self.num_heads) == 1] = attn_logit.reshape(-1)


    attn_matrix = F.softmax(adjacent_matrixes, dim = 2)
    if print_attr:
      print("Attention-probs\n", attn_matrix.permute(0, 3, 1, 2))

      # Corrected einsum equation to match dimensions: b=batch, n=query_node, k=key_node, h=head, f=feature_dim
      attn_matrix = torch.einsum("bnkh,bkhf->bnhf", attn_matrix, node_feats)


    if self.concat_head:
      attn_logit = attn_matrix.reshape(batch_size, num_nodes, -1)
    else:
      attn_logit = attn_logit.mean(dim = 2)

    return attn_logit

In [10]:
layer = GAT(2, 4, num_heads = 4)
# Corrected weight.data to be (4, 2) and bias.data to be (4,)
layer.projection.weight.data = torch.Tensor([[1,0], [1,1], [0,0], [0,0]])
layer.projection.bias.data = torch.Tensor([1,0,0,0])

with torch.no_grad():
  out_feats = layer(node_feats, adj_matrix, print_attr = True)
  print("out_feats\n", out_feats)
  print("adj_matrix\n", adj_matrix)
  print("print_attr\n", True) # Corrected to print the boolean value of print_attr

Attention-probs
 tensor([[[[3.1013e-01, 3.3325e-01, 3.3325e-01, 2.3371e-02],
          [6.0875e-01, 1.6717e-01, 5.6913e-02, 1.6717e-01],
          [8.5018e-01, 5.5528e-02, 3.8764e-02, 5.5528e-02],
          [9.6111e-01, 1.4930e-02, 1.4930e-02, 9.0270e-03]],

         [[1.5799e-01, 4.1803e-01, 4.1803e-01, 5.9574e-03],
          [1.1352e-02, 4.9369e-01, 1.2765e-03, 4.9369e-01],
          [6.9899e-04, 4.9961e-01, 7.8601e-05, 4.9961e-01],
          [4.2560e-05, 4.9998e-01, 4.9998e-01, 1.6049e-06]],

         [[2.5000e-01, 2.5000e-01, 2.5000e-01, 2.5000e-01],
          [2.5000e-01, 2.5000e-01, 2.5000e-01, 2.5000e-01],
          [2.5000e-01, 2.5000e-01, 2.5000e-01, 2.5000e-01],
          [2.5000e-01, 2.5000e-01, 2.5000e-01, 2.5000e-01]],

         [[2.5000e-01, 2.5000e-01, 2.5000e-01, 2.5000e-01],
          [2.5000e-01, 2.5000e-01, 2.5000e-01, 2.5000e-01],
          [2.5000e-01, 2.5000e-01, 2.5000e-01, 2.5000e-01],
          [2.5000e-01, 2.5000e-01, 2.5000e-01, 2.5000e-01]]]])
out_feats
 ten

In [11]:
gcn_layers = {
    "GCN":geom_nn.GCNConv,
    "GAT_LAYER":geom_nn.GATConv,
    "GRAPHCONV_LAYER":geom_nn.GraphConv

}

# Graph-Neural-Network

In [12]:
cora_dataset = torch_geometric.datasets.Planetoid(root = data_path, name = "Cora")
cora_dataset[0]

Data(x=[2708, 1433], edge_index=[2, 10556], y=[2708], train_mask=[2708], val_mask=[2708], test_mask=[2708])

In [13]:
class GNN(nn.Module):
  def __init__(self,c_in, c_hidden, c_out, num_layers = 2, layer_name = "GCN", dp_rate = 0.2, **kwargs):
    super().__init__()

    self.num_layers = num_layers
    gnn_layer = gcn_layers[layer_name]
    in_channels, out_channels = c_in, c_hidden

    layers = []

    for idx in range(num_layers-1):
      layers += [gnn_layer(in_channels = in_channels,
                           out_channels = out_channels,
                            **kwargs),
                 nn.ReLU(inplace = True),
                 nn.Dropout(dp_rate)]
      in_channels = c_hidden

      layers += [gnn_layer(in_channels = in_channels,
                            out_channels = c_out,
                            **kwargs)]
      self.layers = nn.ModuleList(layers)



  def forward(self, x, edge_index):
    for l in self.layers:
      if isinstance(l, geom_nn.MessagePassing):
        x = l(x, edge_index)
      else:
        x = l(x)
    return x

In [14]:
class MLP_Module(nn.Module):
  def __init__(self, c_in, c_hidden, c_out, dp_rate = 0.1, num_layers = 2, **kwargs):
    super().__init__()

    layers = []
    self.num_layers = num_layers
    in_channels, out_channels = c_in, c_hidden

    for idx in range(num_layers-1):
      layers +=[nn.Linear(in_features = in_channels,
                          out_features = out_channels),
                nn.ReLU(inplace = True),
                nn.Dropout(dp_rate)] # Corrected nn.Drop_out to nn.Dropout
      in_channels = c_hidden
      out_channels = c_out

      layers += [nn.Linear(in_features = in_channels, out_features = out_channels)] # Corrected argument names

      self.layers = nn.Sequential(*layers)


  def forward(self, x, *args, **kwargs):
    x = self.layers(x)
    return x

In [15]:
class GCNMODULE(pl.LightningModule):
  def __init__(self, model_name, **model_kwargs):
    super().__init__()

    self.save_hyperparameters()

    if model_name == "MLP_Module":
      self.model = MLP_Module(**model_kwargs)
    else:
      self.model = GNN(**model_kwargs)
    self.loss_entropy = nn.CrossEntropyLoss()


  def forward(self, data, mode = "train"):
    x, edge_index = data.x, data.edge_index
    x = self.model(x, edge_index)


    if mode =="train":
      mask = data.train_mask
    elif mode == "val":
      mask = data.val_mask
    elif mode == "test":
      mask = data.test_mask
    else:
      assert False, f"unknown assert mode {mode}"



    loss = self.loss_entropy(x[mask], data.y[mask])
    acc = (x[mask].argmax(dim = -1)== data.y[mask]).sum().float() / mask.sum()
    return loss, acc




  def configure_optimizers(self):
    optimizer = torch.optim.Adam(self.parameters(), lr = 0.01, weight_decay = 0.01)
    scheduler = torch.optim.lr_scheduler.MultiStepLR(optimizer, milestones = [100, 150], gamma = 0.1)
    return [optimizer], {"scheduler":scheduler, "interval":"step"}



  def training_step(self, batch, batch_idx):
    loss, acc = self.forward(batch, mode = "train")
    self.log("train_loss",loss)
    self.log("train_acc", acc)
    return loss


  def validation_step(self, batch, batch_idx):
    _, acc= self.forward(batch, mode = "val")
    self.log("val_acc", acc)



  def test_step(self, batch, batch_idx):
    _, acc = self.forward(batch, mode = "test")
    self.log("test_acc", acc)



In [16]:
def training_graph_classifier(model_name, dataset, **model_kwargs):
  pl.seed_everything(42)
  node_data_loader = geom_data.DataLoader(dataset, batch_size = 1,shuffle = False)
  root_dir =  os.path.join(checkpoint_path, "Graph_level" + model_name)
  os.makedirs(root_dir, exist_ok = True)
  trainer = pl.Trainer(default_root_dir = root_dir,
                       accelerator = "auto",
                       max_epochs = 90,
                       min_epochs = 10,
                       callbacks = [ModelCheckpoint(save_weights_only = True, mode = "max", monitor = "val_acc"), LearningRateMonitor("epoch")])
  trainer.logger._default_hp_metric = None
  trainer.logger._graph_metric = True

  pretrained_filename = os.path.join(checkpoint_path, f"NodeLevel{model_name}.ckpt")
  if os.path.isfile(pretrained_filename):
    model = GCNMODULE.load_from_checkpoint(pretrained_filename)
  else:
    pl.seed_everything(42)
    model = GCNMODULE(model_name = model_name, c_in = dataset.num_features, c_out = dataset.num_classes, **model_kwargs)
    trainer.fit(model, node_data_loader, node_data_loader)
    model = GCNMODULE.load_from_checkpoint(trainer.checkpoint_callback.best_model_path)
  test_result = trainer.test(model, node_data_loader, verbose = False)
  batch = next(iter(node_data_loader))
  batch = batch.to(model.device)
  _, train_acc = model.forward(batch, mode = "train")
  _, val_acc = model.forward(batch, mode = "val")

  results = {"test_acc":test_result[0]["test_acc"], "train_acc":train_acc, "val_acc":val_acc}
  return model, results



In [17]:
def print_results(result_dict):
    if "train" in result_dict:
        print(f"Train accuracy: {(100.0*result_dict['train']):4.2f}%")
    if "val" in result_dict:
        print(f"Val accuracy:   {(100.0*result_dict['val']):4.2f}%")
    print(f"Test accuracy:  {(100.0*result_dict['test']):4.2f}%")

In [18]:
node_feats_model, node_feats_results = training_graph_classifier(model_name = "MLP_Module", dataset = cora_dataset, c_hidden = 16, num_layers = 2, dp_rate = 0.2, lr = 0.01)

print("node_feats_results", node_feats_results)

INFO:lightning_fabric.utilities.seed:Seed set to 42
/tmp/ipykernel_4378/2849646261.py:3: UserWarning: 'data.DataLoader' is deprecated, use 'loader.DataLoader' instead
  node_data_loader = geom_data.DataLoader(dataset, batch_size = 1,shuffle = False)
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:lightning_fabric.utilities.seed:Seed set to 42
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


┏━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name         ┃ Type             ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ model        │ MLP_Module       │ 23.1 K │ train │     0 │
│ 1 │ loss_entropy │ CrossEntropyLoss │      0 │ train │     0 │
└───┴──────────────┴──────────────────┴────────┴───────┴───────┘

Trainable params: 23.1 K                                                                                           
Non-trainable params: 0                                                                                            
Total params: 23.1 K                                                                                               
Total estimated model params size (MB): 0.092                                                                      
Modules in train mode: 7                                                                                           
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

/usr/local/lib/python3.13/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)`
is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.

...

/usr/local/lib/python3.13/dist-packages/pytorch_lightning/loops/fit_loop.py:321: The number of training batches (1)
is smaller than the logging interval Trainer(log_every_n_steps=50). Set a lower value for log_every_n_steps if you 
want to see logs for the training epoch.

INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=90` reached.


INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Output()

node_feats_results {'test_acc': 0.5759999752044678, 'train_acc': tensor(1.), 'val_acc': tensor(0.4780)}


In [19]:
import inspect

print(inspect.getsource(training_graph_classifier))

def training_graph_classifier(model_name, dataset, **model_kwargs):
  pl.seed_everything(42)
  node_data_loader = geom_data.DataLoader(dataset, batch_size = 1,shuffle = False)
  root_dir =  os.path.join(checkpoint_path, "Graph_level" + model_name)
  os.makedirs(root_dir, exist_ok = True)
  trainer = pl.Trainer(default_root_dir = root_dir,
                       accelerator = "auto",
                       max_epochs = 90,
                       min_epochs = 10,
                       callbacks = [ModelCheckpoint(save_weights_only = True, mode = "max", monitor = "val_acc"), LearningRateMonitor("epoch")])
  trainer.logger._default_hp_metric = None
  trainer.logger._graph_metric = True

  pretrained_filename = os.path.join(checkpoint_path, f"NodeLevel{model_name}.ckpt")
  if os.path.isfile(pretrained_filename):
    model = GCNMODULE.load_from_checkpoint(pretrained_filename)
  else:
    pl.seed_everything(42)
    model = GCNMODULE(model_name = model_name, c_in = dataset.num_features, c_out

In [20]:
node_feats_model, node_feats_results = training_graph_classifier(model_name = "GCNMODULE", dataset = cora_dataset, c_hidden = 16, num_layers = 2, dp_rate = 0.2,)

print("node_feats_results", node_feats_results)

INFO:lightning_fabric.utilities.seed:Seed set to 42
/tmp/ipykernel_4378/2849646261.py:3: UserWarning: 'data.DataLoader' is deprecated, use 'loader.DataLoader' instead
  node_data_loader = geom_data.DataLoader(dataset, batch_size = 1,shuffle = False)
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:lightning_fabric.utilities.seed:Seed set to 42
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


┏━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name         ┃ Type             ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ model        │ GNN              │ 23.1 K │ train │     0 │
│ 1 │ loss_entropy │ CrossEntropyLoss │      0 │ train │     0 │
└───┴──────────────┴──────────────────┴────────┴───────┴───────┘

Trainable params: 23.1 K                                                                                           
Non-trainable params: 0                                                                                            
Total params: 23.1 K                                                                                               
Total estimated model params size (MB): 0.092                                                                      
Modules in train mode: 11                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=90` reached.


INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Output()

node_feats_results {'test_acc': 0.8029999732971191, 'train_acc': tensor(0.9929), 'val_acc': tensor(0.7400)}


In [21]:
tu_dataset = torch_geometric.datasets.TUDataset(root = data_path, name = "MUTAG")

In [22]:
tu_dataset[0]

Data(edge_index=[2, 38], x=[17, 7], edge_attr=[38, 4], y=[1])

In [23]:
print("tu_dataset", tu_dataset.data)
print("len(tu_dataset", len(tu_dataset))
print(f"Average label: {tu_dataset.data.y.float().mean().item():4.2f}")

tu_dataset Data(x=[3371, 7], edge_index=[2, 7442], edge_attr=[7442, 4], y=[188])
len(tu_dataset 188
Average label: 0.66


/tmp/ipykernel_4378/1578353710.py:1: UserWarning: It is not recommended to directly access the internal storage format `data` of an 'InMemoryDataset'. The data of the dataset is already cached, so any modifications to `data` will not be reflected when accessing its elements. Clearing the cache now by removing all elements in `dataset._data_list`. If you are absolutely certain what you are doing, access the internal storage via `InMemoryDataset._data` instead to suppress this warning. Alternatively, you can access stacked individual attributes of every graph via `dataset.{attr_name}`.
  print("tu_dataset", tu_dataset.data)
/tmp/ipykernel_4378/1578353710.py:3: UserWarning: It is not recommended to directly access the internal storage format `data` of an 'InMemoryDataset'. If you are absolutely certain what you are doing, access the internal storage via `InMemoryDataset._data` instead to suppress this warning. Alternatively, you can access stacked individual attributes of every graph via 

In [24]:
torch.manual_seed(42)
tu_dataset.shuffle()
train_dataset = tu_dataset[:150]
test_dataset = tu_dataset[150:]

train_dataset
test_dataset


MUTAG(38)

In [25]:
print("train_dataset", train_dataset)
print("test_dataset", test_dataset)

train_dataset MUTAG(150)
test_dataset MUTAG(38)


In [26]:
train_loader = geom_data.DataLoader(train_dataset, batch_size = 64, shuffle = True)
val_loader = geom_data.DataLoader(test_dataset, batch_size = 64)
test_loader = geom_data.DataLoader(test_dataset, batch_size = 64)


/tmp/ipykernel_4378/3167935547.py:1: UserWarning: 'data.DataLoader' is deprecated, use 'loader.DataLoader' instead
  train_loader = geom_data.DataLoader(train_dataset, batch_size = 64, shuffle = True)
/tmp/ipykernel_4378/3167935547.py:2: UserWarning: 'data.DataLoader' is deprecated, use 'loader.DataLoader' instead
  val_loader = geom_data.DataLoader(test_dataset, batch_size = 64)
/tmp/ipykernel_4378/3167935547.py:3: UserWarning: 'data.DataLoader' is deprecated, use 'loader.DataLoader' instead
  test_loader = geom_data.DataLoader(test_dataset, batch_size = 64)


In [27]:
class Graph_model(nn.Module):
  def __init__(self, c_in, c_hidden, c_out, dp_rate_linear = 0.5, num_layers=None, layer_name=None, **kwargs):
    super().__init__()
    self.layers = GNN(c_in = c_in,
                      c_hidden = c_hidden,
                      c_out = c_hidden,
                      num_layers = num_layers,
                      layer_name = layer_name,
                      **kwargs
                      )

    self.linear = nn.Sequential(
        nn.Dropout(dp_rate_linear),
        nn.Linear(c_hidden, c_out)
    )

  def forward(self, x, edge_index, batch_idx):
    x = self.layers(x, edge_index)
    x = geom_nn.global_mean_pool(x, batch_idx)
    x = self.linear(x)
    return x

In [28]:
class graph_level(pl.LightningModule):
  def __init__(self, **model_kwargs):
    super().__init__()
    self.save_hyperparameters()
    self.model = Graph_model(**model_kwargs)
    self.loss = nn.BCEWithLogitsLoss() if self.hparams.c_out == 1 else nn.CrossEntropyLoss()

  def forward(self, data, mode = "train"):
    x, edge_index, batch_idx = data.x, data.edge_index, data.batch
    x = self.model(x, edge_index, batch_idx)

    if self.hparams.c_out == 1:
      x = x.squeeze(dim = -1) # Squeeze only for binary classification
      preds = ( x > 0).float()
      data.y = data.y.float()
    else:
      # For multi-class, x should already be (batch_size, num_classes)
      data.y = data.y.long()
      preds = x.argmax(dim = -1)

    # Moved outside the if/else block to ensure a return value in all cases
    loss = self.loss(x, data.y)
    acc = (preds == data.y).sum().float() / preds.shape[0]
    return loss, acc

  def configure_optimizers(self):
    optimizer = torch.optim.AdamW(self.parameters(), lr = 1e-3, weight_decay = 0.0)
    scheduler = torch.optim.lr_scheduler.MultiStepLR(optimizer, milestones = [100, 150], gamma = 0.1)
    return [optimizer], {"scheduler":scheduler, "interval":"epoch"}

  def training_step(self, batch, batch_idx):
    loss, acc = self.forward(batch, mode = "train")
    self.log("train_loss", loss)
    self.log("train_acc", acc)
    return loss # Typically, training_step returns only loss

  def validation_step(self, batch, batch_idx):
    _, acc = self.forward(batch, mode = "val")
    self.log("val_acc", acc)

  def test_step(self, batch, batch_idx):
    _, acc = self.forward(batch, mode = "test")
    self.log("test_acc", acc)

In [29]:
def training_graph(model_name, **model_kwargs):
  pl.seed_everything(42)
  root_dir = os.path.join(checkpoint_path, "Graph_level" + model_name)
  trainer = pl.Trainer(default_root_dir = root_dir,
                       max_epochs = 100,
                       min_epochs = 10,
                       callbacks = [ModelCheckpoint(save_weights_only = True,
                                                    mode = "max", monitor = "val_acc"), LearningRateMonitor("epoch")])
  trainer.logger._default_hp_metric = None
  trainer.logger._graph_metric = True

  pretrained_filename = os.path.join(checkpoint_path, f"GraphLevel{model_name}.ckpt")
  if  os.path.isfile(pretrained_filename):
    model = graph_level.load_from_checkpoint_path(pretrained_filename)
  else:
    pl.seed_everything(42)
    model = graph_level(c_in = tu_dataset.num_features, c_out = 1 if tu_dataset.num_classes==2 else tu_dataset.num_classes, **model_kwargs)
    trainer.fit(model, train_loader, val_loader)
    model = graph_level.load_from_checkpoint(trainer.checkpoint_callback.best_model_path)
  train_results = trainer.test(model, train_loader, verbose = False)
  test_results = trainer.test(model, test_loader, verbose = False)
  val_results = trainer.validate(model, val_loader, verbose = False)
  result = {"test_acc":test_results[0]["test_acc"], "train_acc":train_results[0]["test_acc"],"val_acc":val_results[0]["val_acc"]}
  model = model.to(device)
  return model, result

In [30]:
trainer_model, trainer_results = training_graph( model_name = "GRAPHCONV_LAYER",
                                                layer_name = "GRAPHCONV_LAYER",
                                                c_hidden = 256,
                                                dp_rate_linear = 0.5,
                                                 num_layers = 3)

print("trainer_results", trainer_results)

INFO:lightning_fabric.utilities.seed:Seed set to 42
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:lightning_fabric.utilities.seed:Seed set to 42
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


┏━━━┳━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name  ┃ Type              ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ model │ Graph_model       │  398 K │ train │     0 │
│ 1 │ loss  │ BCEWithLogitsLoss │      0 │ train │     0 │
└───┴───────┴───────────────────┴────────┴───────┴───────┘

Trainable params: 398 K                                                                                            
Non-trainable params: 0                                                                                            
Total params: 398 K                                                                                                
Total estimated model params size (MB): 1.592                                                                      
Modules in train mode: 27                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

/usr/local/lib/python3.13/dist-packages/pytorch_lightning/utilities/data.py:79: Trying to infer the `batch_size` 
from an ambiguous collection. The batch size we found is 2. To avoid any miscalculations, use `self.log(..., 
batch_size=batch_size)`.

...

INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Output()

INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Output()

trainer_results {'test_acc': 0.8684210777282715, 'train_acc': 0.8555871844291687, 'val_acc': 0.8684210777282715}
